# 03 - Advanced models, tuning and experiment tracking

1. Keras MLP (29 -> 64 -> 32 -> 1 with dropout) and an XGBoost model (first attempt, `src/advanced.py`)
2. Hyperparameter tuning of XGBoost, Random Forest and the MLP (`src/tune.py`)
3. Experiment log (`reports/experiments.csv`)

Everything here uses train and validation only. The test set is not touched in this notebook.

In [ ]:
import os
# the notebooks live in notebooks/, but all paths below are from the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.width", 220, "display.max_columns", None, "display.max_colwidth", 90)

# Set to True to re-run (several minutes for the MLP search)
RERUN = False
if RERUN:
    import runpy, sys
    runpy.run_path("src/advanced.py")
    for mode in ["xgb", "rf", "mlp", "compare"]:
        sys.argv = ["tune.py", mode]
        runpy.run_path("src/tune.py", run_name="__main__")

## Part 1 - First MLP and XGBoost

Design choices: small network because there are only 331 training frauds; class weight sqrt(neg/pos) = 24.5
instead of the full ratio (about 600) which makes training noisy; early stopping on validation PR-AUC;
3 random seeds because one run can be lucky.

In [ ]:
adv = pd.read_csv("reports/advanced_results.csv")
adv[["model", "PR_AUC", "ROC_AUC", "precision@best", "recall@best", "F1@best", "FP@best", "FN@best", "latency_ms"]].round(3)

In [ ]:
display(Image("reports/figures/mlp_training_curves.png"))

Training curves: training PR-AUC keeps climbing (to about 0.89) while validation climbs slower (about 0.83) and
is very jagged from epoch to epoch (+-0.02 to 0.03). That is mild overfitting plus a small validation set.

**The first MLP was unstable.** The three seeds gave PR-AUC 0.697, 0.833 and 0.702 (mean 0.744, std 0.063).
Seeds 42 and 44 were stopped after only 14 epochs because early stopping (patience 10) locked on to a
noisy early peak. So 0.833 was a lucky seed and not the real performance of that setup.

**The first XGBoost was under-trained.** It stopped at iteration 16 (early stopping on the noisy validation set),
giving PR-AUC 0.735. That says more about my setup than about XGBoost, so it was tuned properly next.

## Part 2 - Tuning

- **XGBoost** (15 random configs) and **Random Forest** (6 configs): 3-fold stratified cross-validation on the training
  set, scored by average precision, so the validation set is not used for early stopping.
- **MLP** (9 configs x 3 seeds): patience 30, lower learning rates, configs ranked by the **mean** over seeds
  (not the best seed). The final MLP is an average of the 3 seeds.

In [ ]:
log = pd.read_csv("reports/experiments.csv")
if "cv_pr_auc" not in log.columns:
    raise RuntimeError("Run  python src/clean_experiments_log.py  first, then re-run this cell")
print("Logged runs:", len(log))
log.groupby("model").size()

### XGBoost: cross-validation results

In [ ]:
x = log[log["model"] == "XGBoost"].sort_values("cv_pr_auc", ascending=False)
x[["config", "cv_pr_auc", "cv_std"]].head(8)

### Random Forest: cross-validation results

In [ ]:
r = log[log["model"] == "RandomForest"].sort_values("cv_pr_auc", ascending=False)
r[["config", "cv_pr_auc", "cv_std"]]

### MLP: mean and std over 3 seeds for each configuration

In [ ]:
m = log[log["model"] == "MLP"]
summ = (m.groupby("config")["val_pr_auc"].agg(["mean", "std", "count"])
          .sort_values("mean", ascending=False).round(4))
display(summ)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(summ)), summ["mean"], yerr=summ["std"], capsize=4)
ax.set_xlabel("MLP configuration (best to worst)"); ax.set_ylabel("val PR-AUC (mean +- std over 3 seeds)")
ax.set_title("MLP tuning: stability across seeds")
plt.tight_layout(); plt.savefig("reports/figures/mlp_tuning_summary.png", dpi=150); plt.show()

### Best runs

In [ ]:
log[log["model"].str.contains("best", na=False)][["model", "config", "seed", "val_pr_auc", "val_roc_auc"]]

## Results after tuning (validation set)

| Model | Val PR-AUC | ROC-AUC | How it was selected |
|---|---|---|---|
| MLP (3-seed ensemble) | 0.858 | 0.958 | config picked on validation, early-stopped on validation |
| XGBoost (tuned) | 0.835 | 0.970 | 3-fold CV on train (CV score 0.856) |
| Random Forest (tuned) | 0.831 | 0.960 | 3-fold CV on train (CV score 0.839) |
| Random Forest (untuned baseline) | 0.855 | 0.947 | none |

Best MLP: learning rate 3e-4, batch 2048, layers (128, 64), dropout 0.2, class weight 10.
Per-seed PR-AUC 0.844 / 0.865 / 0.861 (std 0.009), compared to a std of 0.063 before tuning, so a lower
learning rate and a smaller class weight fixed the instability. XGBoost tuned best at depth 5, 600 trees,
learning rate 0.1, scale_pos_weight 24.5.

**Notes**

- XGBoost recovered from 0.735 to 0.835 once early stopping on the noisy validation set was removed. Its CV
  score is close to its validation score, so the tuning can be trusted.
- The tuned Random Forest (0.831) is not better than the untuned one (0.855). That gap is noise, so tuning gave
  no reliable gain for Random Forest.
- The MLP's lead on validation is partly selection bias (best of 9 configs, picked and early-stopped on the same
  71 frauds). With this few frauds, differences below about 0.02-0.03 PR-AUC are noise, so the models are
  effectively tied here. The test set (see the evaluation report) is the fair comparison.
- **Threshold rule:** maximum recall subject to validation precision >= 0.90. The max-F1 threshold for the MLP
  gave 0 false positives on validation, which looks perfect but is a knife-edge value overfitted to 71 frauds,
  so it is not used.